# Matrix Factorization for Recommendation Systems

## Introduction

Recommendation systems are widely used by streaming platforms and e-commerce applications to provide personalized suggestions to users. In this notebook, we build a recommendation system using Matrix Factorization, one of the most powerful collaborative filtering techniques.

The project focuses on decomposing the user-item interaction matrix into latent feature representations to better understand user preferences and item similarities. The notebook includes data preprocessing, matrix decomposition, model training, and recommendation generation.

In [1]:
import pandas as pd

In [9]:
ratings=pd.read_csv('/kaggle/input/datasets/nihalba/u-data/u.data',sep='\t', header=None)

In [10]:
ratings

,0,1,2,3
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [11]:
ratings=pd.read_csv('/kaggle/input/datasets/nihalba/u-data/u.data',sep='\t', header=None, names=['user_id','movie_id','rating'],usecols=range(3)) #ilk 3 sutun sectik

In [12]:
ratings

,user_id,movie_id,rating
0,196,242,3
1,186,302,3
2,22,377,1
3,244,51,2
4,166,346,1
...,...,...,...
99995,880,476,3
99996,716,204,5
99997,276,1090,1
99998,13,225,2


In [13]:
movies=pd.read_csv('/kaggle/input/datasets/nihalba/u-item/u.item',encoding="iso-8859-1",sep="|",header=None,names=["movie_id","title"],usecols=range(2))

In [14]:
movies.head()

,movie_id,title
0,1,Toy Story (1995)
1,2,GoldenEye (1995)
2,3,Four Rooms (1995)
3,4,Get Shorty (1995)
4,5,Copycat (1995)


In [15]:
user = pd.read_csv('/kaggle/input/datasets/nihalba/u-user/u.user', sep='|', header=None, names=['user_id', 'Age', 'Gender', 'Profession', 'Zipcode'])

In [16]:
user

,user_id,Age,Gender,Profession,Zipcode
0,1,24,M,technician,85711
1,2,53,F,other,94043
2,3,23,M,writer,32067
3,4,24,M,technician,43537
4,5,33,F,other,15213
...,...,...,...,...,...
938,939,26,F,student,33319
939,940,32,M,administrator,02215
940,941,20,M,student,97229
941,942,48,F,librarian,78209


In [17]:
rating=pd.merge(movies,ratings)

In [18]:
rating.head()

,movie_id,title,user_id,rating
0,1,Toy Story (1995),308,4
1,1,Toy Story (1995),287,5
2,1,Toy Story (1995),148,4
3,1,Toy Story (1995),280,4
4,1,Toy Story (1995),66,3


In [19]:
rating=pd.merge(rating,user)

In [19]:
rating

,movie_id,title,user_id,rating,Age,Gender,Profession,Zipcode
0,1,Toy Story (1995),308,4,60,M,retired,95076
1,1,Toy Story (1995),287,5,21,M,salesman,31211
2,1,Toy Story (1995),148,4,33,M,engineer,97006
3,1,Toy Story (1995),280,4,30,F,librarian,22903
4,1,Toy Story (1995),66,3,23,M,student,80521
...,...,...,...,...,...,...,...,...
99995,1678,Mat' i syn (1997),863,1,17,M,student,60089
99996,1679,B. Monkey (1998),863,3,17,M,student,60089
99997,1680,Sliding Doors (1998),863,2,17,M,student,60089
99998,1681,You So Crazy (1994),896,3,28,M,writer,91505


In [20]:
rating.head()

,movie_id,title,user_id,rating,Age,Gender,Profession,Zipcode
0,1,Toy Story (1995),308,4,60,M,retired,95076
1,1,Toy Story (1995),287,5,21,M,salesman,31211
2,1,Toy Story (1995),148,4,33,M,engineer,97006
3,1,Toy Story (1995),280,4,30,F,librarian,22903
4,1,Toy Story (1995),66,3,23,M,student,80521


In [21]:
rating.title.value_counts()

title
Star Wars (1977)                        583
Contact (1997)                          509
Fargo (1996)                            508
Return of the Jedi (1983)               507
Liar Liar (1997)                        485
                                       ... 
Invitation, The (Zaproszenie) (1986)      1
Symphonie pastorale, La (1946)            1
Nothing Personal (1995)                   1
Ripe (1996)                               1
Brother's Kiss, A (1997)                  1
Name: count, Length: 1664, dtype: int64

In [22]:
mf=rating.pivot_table(index=['user_id'],columns=['title'],values='rating')

In [23]:
mf.head()

title,'Til There Was You (1997),1-900 (1994),101 Dalmatians (1996),12 Angry Men (1957),187 (1997),2 Days in the Valley (1996),"20,000 Leagues Under the Sea (1954)",2001: A Space Odyssey (1968),3 Ninjas: High Noon At Mega Mountain (1998),"39 Steps, The (1935)",...,Yankee Zulu (1994),Year of the Horse (1997),You So Crazy (1994),Young Frankenstein (1974),Young Guns (1988),Young Guns II (1990),"Young Poisoner's Handbook, The (1995)",Zeus and Roxanne (1997),unknown,Á köldum klaka (Cold Fever) (1994)
user_id,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,2.0,5.0,NaN,NaN,3.0,4.0,NaN,NaN,...,NaN,NaN,NaN,5.0,3.0,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,2.0,NaN,NaN,NaN,NaN,4.0,NaN,NaN,...,NaN,NaN,NaN,4.0,NaN,NaN,NaN,NaN,4.0,NaN


In [24]:
swr=mf['Star Wars (1977)']

In [25]:
swr.value_counts()

Star Wars (1977)
5.0    325
4.0    176
3.0     57
2.0     16
1.0      9
Name: count, dtype: int64

In [26]:
mf[['Star Wars (1977)','101 Dalmatians (1996)']].corr()

title,Star Wars (1977),101 Dalmatians (1996)
title,,
Star Wars (1977),1.000000,0.211132
101 Dalmatians (1996),0.211132,1.000000


In [27]:
sm=mf.corrwith(swr)

/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2914: RuntimeWarning: Degrees of freedom <= 0 for slice
  c = cov(x, y, rowvar, dtype=dtype)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: divide by zero encountered in divide
  c *= np.true_divide(1, fact)
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2773: RuntimeWarning: invalid value encountered in multiply
  c *= np.true_divide(1, fact)


In [28]:
sm.sort_values(ascending=False)

title
Hollow Reed (1996)                         1.0
Stripes (1981)                             1.0
No Escape (1994)                           1.0
Man of the Year (1995)                     1.0
Cosi (1996)                                1.0
                                          ... 
Wonderland (1997)                          NaN
Wooden Man's Bride, The (Wu Kui) (1994)    NaN
Yankee Zulu (1994)                         NaN
You So Crazy (1994)                        NaN
Á köldum klaka (Cold Fever) (1994)         NaN
Length: 1664, dtype: float64

In [29]:
import numpy as np

In [30]:
mstats=rating.groupby('title').agg({'rating':[np.size,np.mean]}) #kac kisi oy verdi, ve ortalamasi

/tmp/ipykernel_57/2702752719.py:1: FutureWarning: The provided callable <function mean at 0x7ced405bd300> is currently using SeriesGroupBy.mean. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "mean" instead.
  mstats=rating.groupby('title').agg({'rating':[np.size,np.mean]}) #kac kisi oy verdi, ve ortalamasi


In [31]:
mstats.head()

rating          
                            size      mean
title                                     
'Til There Was You (1997)      9  2.333333
1-900 (1994)                   5  2.600000
101 Dalmatians (1996)        109  2.908257
12 Angry Men (1957)          125  4.344000
187 (1997)                    41  3.024390

In [32]:
pm=mstats[mstats['rating']['size']>100] # en az 100 kisi oy vermis olsun

In [33]:
smdata=pd.DataFrame(sm,columns=['similarity'])

In [34]:
pm.columns=pm.columns.get_level_values(0)
df=pm.join(smdata)

In [35]:
df.sort_values(['similarity'],ascending=False)

,rating,rating,similarity
title,,,
Star Wars (1977),583,4.358491,1.000000
"Empire Strikes Back, The (1980)",367,4.204360,0.747981
Return of the Jedi (1983),507,4.007890,0.672556
Raiders of the Lost Ark (1981),420,4.252381,0.536117
Austin Powers: International Man of Mystery (1997),130,3.246154,0.377433
...,...,...,...
"Edge, The (1997)",113,3.539823,-0.127167
As Good As It Gets (1997),112,4.196429,-0.130466
Crash (1996),128,2.546875,-0.148507


## Conclusion

In this notebook, a recommendation system was successfully developed using Matrix Factorization techniques. By decomposing the user-item interaction matrix into latent factors, the model was able to capture hidden relationships between users and items.

The project demonstrates the effectiveness of collaborative filtering methods for personalized recommendation systems. Matrix Factorization helps reduce sparsity issues while improving recommendation quality through latent feature learning.

Overall, this notebook provides a strong foundation for understanding recommendation systems, collaborative filtering, and matrix decomposition techniques commonly used in real-world applications such as Netflix, Spotify, and Amazon recommendation engines.